In [1]:
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.ui.workspace import CloudWorkspace

## Configuration credential

In [2]:
# Configuration for the credential
import os
from dotenv import load_dotenv

ENV_PATH = ".env"
load_dotenv(ENV_PATH)

False

## Create project on evidently AI

In [3]:
ws = CloudWorkspace(token=os.getenv("EVIDENTLY_TOKEN"), url="https://app.evidently.cloud")

## Create a Project within your Organization, or connect to an existing Project:

In [4]:
try:
    existing_projects = ws.list_projects()
    project = next((p for p in existing_projects if p.name == "LLM Audit"), None)
    
    if project is None:
        project = ws.create_project(name="LLM Audit", org_id="019d6885-06c6-7601-9a3e-2298bed991f4")
        project.description = "Project for auditing LLM performance on models, data, and LLMs evaluations."
        project.save()
        print(f"✅ Project created: {project.name}")   # ← print, not project()
    else:
        print(f"📁 Project already exists: {project.name}")  # ← print, not project()

except Exception as e:
    print(f"Error accessing or creating project: {e}")

print(f"Project name: {project.name}")

📁 Project already exists: LLM Audit
Project name: LLM Audit


## Prepare the dataset -> Merge prompt with dataframe into reliable data information
* ### Load dataset
* ### Create prompt

## Load dataset

In [5]:
pd.set_option('display.max_columns', None)
df = pd.read_parquet("../../Database/data/eda_banking.parquet")
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,11983969.0,1,1,1,10134888.0,1,1,2,DIAMOND,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,8380786.0,1,0,1,11254258.0,0,1,3,DIAMOND,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,15966080.0,3,1,0,11393157.0,1,1,3,DIAMOND,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,11983969.0,2,0,0,9382663.0,0,0,5,GOLD,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,12551082.0,1,1,1,7908410.0,0,0,5,GOLD,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,11983969.0,2,1,0,9627064.0,0,0,1,DIAMOND,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,5736961.0,1,1,1,10169977.0,0,0,5,PLATINUM,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,11983969.0,1,0,1,4208558.0,1,1,3,SILVER,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,7507531.0,2,1,0,9288852.0,1,1,2,GOLD,339,1,2,25025.103333,0,0,0,1,1,1,3


## Create LLMs prompt for evidentlyAI production

In [6]:
# Define questions and context
question_and_context = [
    [
        "What is the fraud detection rate in the dataset?",
        "Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical."
    ],
    [
        "Which ML model performs best for credit default prediction?",
        "XGBoost and Random Forest outperform logistic regression on imbalanced datasets."
    ],
    [
        "How should missing values be handled in operational banking data?",
        "Imputation must preserve regulatory compliance and not introduce synthetic bias."
    ],
    [
        "What is the churn rate and how can it be reduced?",
        "Customer churn in retail banking averages 15-20% annually."
    ],
    [
        "Explain the class imbalance issue in fraud detection.",
        "Fraud datasets are heavily imbalanced with fraud rates below 1-2%."
    ],
]

pd.set_option('display.max_colwidth', None)
eval_df = pd.DataFrame(question_and_context, columns=["question", "context"])

In [7]:
eval_df.head()

,question,context
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.
3,What is the churn rate and how can it be reduced?,Customer churn in retail banking averages 15-20% annually.
4,Explain the class imbalance issue in fraud detection.,Fraud datasets are heavily imbalanced with fraud rates below 1-2%.


In [8]:
from groq import Groq
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

client = Groq(api_key=os.environ.get("GROQ_API_KEY_2"))

# 1. Build data context from dataframe
def build_data_context(df):
    context = f"""
You are a banking data analyst & data scentist. Use the following dataset information to reasong and answer questions.

DATASET OVERVIEW:
- Total Records: {len(df):,}
- Columns: {list(df.columns)}

STATISTICAL SUMMARY:
{df.describe().to_string()}

CATEGORICAL COLUMNS:
- Geography Distribution: {df['Geography'].value_counts().to_dict()}
- Gender Distribution: {df['Gender'].value_counts().to_dict()}
- Card Type Distribution: {df['Card Type'].value_counts().to_dict()}

- KEY METRICS:
- Fraud Rate: {df['Fraud'].mean():.2%}
- Churn Rate (Exited): {df['Exited'].mean():.2%}
- Avg Credit Score: {df['CreditScore'].mean():.2f}
- Avg Balance : {df['Balance'].mean():.2f}
- Avg Risk Score: {df['RiskScore'].mean():.2f}
- High Value Customers: {df['HighValueCustomer'].sum():,} ({df['HighValueCustomer'].mean():.2%})
- Customers with Complaints: {df['Complain'].sum():,} ({df['Complain'].mean():.2%})
- Active Members: {df['IsActiveMember'].sum():,} ({df['IsActiveMember'].mean():.2%})

CORRELATION WITH FRAUD (top features):
{df.corr(numeric_only=True)['Fraud'].sort_values(ascending=False).to_string()}

CORRELATION WITH CHURN (top features):
{df.corr(numeric_only=True)['Exited'].sort_values(ascending=False).to_string()}
"""
    
    return context

data_context = build_data_context(df)


In [9]:

# 2. Generate answers with data-aware prompting
def get_groq_response_with_context(row):
    prompt = f"""
{data_context}

---
CONTEXT HINT: {row['context']}
QUESTION: {row['question']}

Answer based strictly on the dataset statistics provided above. Be specific with numbers and percentages from the data.
"""
    
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1024,
        top_p=0.9,
        stream=False
    )
    return completion.choices[0].message.content

In [10]:
# apply to eval_df
eval_df['answer'] = eval_df.apply(get_groq_response_with_context, axis=1)
eval_df

question  \
0                   What is the fraud detection rate in the dataset?   
1        Which ML model performs best for credit default prediction?   
2  How should missing values be handled in operational banking data?   
3                  What is the churn rate and how can it be reduced?   
4              Explain the class imbalance issue in fraud detection.   

                                                                                    context  \
0  Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.   
1          XGBoost and Random Forest outperform logistic regression on imbalanced datasets.   
2          Imputation must preserve regulatory compliance and not introduce synthetic bias.   
3                                Customer churn in retail banking averages 15-20% annually.   
4                        Fraud datasets are heavily imbalanced with fraud rates below 1-2%.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## Auditing the LLMs for evaluation

* ###  Sentiment: from -1 (negative) to 1 (positive)
* ### Text length: character count
* ### Denials: refusals to answer. This uses an LLM-as-a-judge with built-in prompt.

### Each evaluation is a descriptor. It adds a new score or label to each row in your dataset. For LLM-as-a-judge, we’ll use OpenAI GPT-4o mini. Set OpenAI key as an environment variable:

In [11]:
# Turn on debug
import litellm

litellm._turn_on_debug()

os.environ["GROQ_API_KEY_2"] = os.environ.get("GROQ_API_KEY_2", "")

response = litellm.completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say OK"}],
)
print(response.choices[0].message.content)

13:33:40 - LiteLLM:DEBUG: utils.py:481 - 

13:33:40 - LiteLLM:DEBUG: utils.py:481 - Request to litellm:
13:33:40 - LiteLLM:DEBUG: utils.py:481 - litellm.completion(model='groq/llama-3.1-8b-instant', messages=[{'role': 'user', 'content': 'Say OK'}])
13:33:40 - LiteLLM:DEBUG: utils.py:481 - 

13:33:40 - LiteLLM:DEBUG: utils.py:1007 - Removing thought signatures from tool call IDs for non-Gemini model
13:33:40 - LiteLLM:DEBUG: litellm_logging.py:557 - self.optional_params: {}
13:33:40 - LiteLLM:DEBUG: utils.py:481 - SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
13:33:40 - LiteLLM:DEBUG: utils.py:5605 - checking potential_model_names in litellm.model_cost: {'split_model': 'llama-3.1-8b-instant', 'combined_model_name': 'groq/llama-3.1-8b-instant', 'stripped_model_name': 'llama-3.1-8b-instant', 'combined_stripped_model_name': 'groq/llama-3.1-8b-instant', 'custom_llm_provider': 'groq'}
13:33:40 - LiteLLM:DEBUG: utils.py:5605 - checking potential_mode

OK.


13:33:40 - LiteLLM:DEBUG: litellm_logging.py:1540 - response_cost: 2.09e-06
13:33:40 - LiteLLM:DEBUG: utils.py:5605 - checking potential_model_names in litellm.model_cost: {'split_model': 'llama-3.1-8b-instant', 'combined_model_name': 'groq/llama-3.1-8b-instant', 'stripped_model_name': 'llama-3.1-8b-instant', 'combined_stripped_model_name': 'groq/llama-3.1-8b-instant', 'custom_llm_provider': 'groq'}
13:33:40 - LiteLLM:DEBUG: utils.py:5605 - checking potential_model_names in litellm.model_cost: {'split_model': 'llama-3.1-8b-instant', 'combined_model_name': 'groq/llama-3.1-8b-instant', 'stripped_model_name': 'llama-3.1-8b-instant', 'combined_stripped_model_name': 'groq/llama-3.1-8b-instant', 'custom_llm_provider': 'groq'}


In [17]:
import litellm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY_2", "")
print(f"Key loaded: {bool(os.environ.get('GROQ_API_KEY_2'))}")

# ── Option 1: Tell LiteLLM to auto-retry with backoff ─────────────────────
litellm.num_retries = 5
litellm.retry_after = 10  # wait 10s between retries

eval_dataset = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        Sentiment("answer", alias="Sentiment"),
        TextLength("answer", alias="Length"),
        DeclineLLMEval(
            "answer",
            alias="Denials",
            provider="litellm",
            model="groq/llama-3.3-70b-versatile"
        ),
    ]
)

eval_dataset.as_dataframe()

13:36:14 - LiteLLM:DEBUG: utils.py:481 - 

13:36:14 - LiteLLM:DEBUG: utils.py:481 - Request to litellm:
13:36:14 - LiteLLM:DEBUG: utils.py:481 - litellm.acompletion(model='groq/llama-3.3-70b-versatile', custom_llm_provider=None, messages=[{'role': 'system', 'content': 'You are a judge which evaluates text.'}, {'role': 'user', 'content': 'A "DECLINE" typically refers to a refusal or a polite rejection to do something.\nIn these contexts, "DECLINE" signifies a respectful or formal way of saying no to provide a help, service, or answer.\nClassify text between ___text_starts_here___ and ___text_ends_here___ into two categories: DECLINE and OK.\n___text_starts_here___\nTo determine the fraud detection rate in the dataset, we need to look at the statistics provided. The fraud rate is explicitly mentioned in the KEY METRICS section:\n\n- Fraud Rate: 3.32%\n\nThis means that out of the total records (10,000), 3.32% of the customers are identified as fraudulent. To find the number of fraudulent

Key loaded: True


13:36:14 - LiteLLM:DEBUG: litellm_logging.py:1223 - RAW RESPONSE:
{"id": "chatcmpl-bc6846d6-20ef-4579-946c-91ca0aa837f6", "object": "chat.completion", "created": 1775799375, "model": "llama-3.3-70b-versatile", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\"category\": \"DECLINE\", \"reasoning\": \"The text politely refuses to provide a detailed analysis of the fraud detection rate, stating that more specific information is needed to calculate precision, recall, and F1 score, which signifies a respectful or formal way of saying no to provide a help or service.\"}"}, "logprobs": null, "finish_reason": "stop"}], "usage": {"queue_time": 0.044776289, "prompt_tokens": 502, "prompt_time": 0.03404223, "completion_tokens": 64, "completion_time": 0.183386737, "total_tokens": 566, "total_time": 0.217428967}, "usage_breakdown": null, "system_fingerprint": "fp_dae98b5ecb", "x_groq": {"id": "req_01knty855eekasmv2dg764jb02", "seed": 1154431542}, "service_tier": "on_demand"}


question  \
0                   What is the fraud detection rate in the dataset?   
1        Which ML model performs best for credit default prediction?   
2  How should missing values be handled in operational banking data?   
3                  What is the churn rate and how can it be reduced?   
4              Explain the class imbalance issue in fraud detection.   

                                                                                    context  \
0  Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.   
1          XGBoost and Random Forest outperform logistic regression on imbalanced datasets.   
2          Imputation must preserve regulatory compliance and not introduce synthetic bias.   
3                                Customer churn in retail banking averages 15-20% annually.   
4                        Fraud datasets are heavily imbalanced with fraud rates below 1-2%.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## Deploy to evidentlyAI UI Dashboard

In [18]:
from evidently.ui.workspace import DashboardPanelPlot, DashboardModel, DashboardTabModel
from evidently.sdk.models import PanelMetric

# 1. Define panels with correct PanelMetric fields
panel_sentiment = DashboardPanelPlot(
    title="Answer Sentiment",
    size="full",
    values=[
        PanelMetric(
            metric="Sentiment",
            legend="Sentiment Score",
        )
    ],
    plot_params={"plot_type": "line"}
)

panel_length = DashboardPanelPlot(
    title="Answer Text Length",
    size="full",
    values=[
        PanelMetric(
            metric="Denials",
            legend="Denial Rate"
        )
    ],
    plot_params={"plot_type": "bar"}
)

panel_denials = DashboardPanelPlot(
    title="LLM Denial Rate",
    size="full",
    values=[
        PanelMetric(
            metric="Denials",
            legend="Denial Rate"
        )
    ],
    plot_params={"plot_type": "bar"}
)

# 2. Create tab grouping all panels
tab = DashboardTabModel(
    title="LLM Audit Overview",
    panels=[panel_sentiment.id, panel_length.id, panel_denials.id]
)

# 3. Build dashboard model
dashboard = DashboardModel(
    tabs=[tab],
    panels=[panel_sentiment, panel_length, panel_denials]
)

# 4. Push to Evidently Cloud
ws.save_dashboard(project.id, dashboard)
print("✅ Dashboard saved to Evidently Cloud")
print(f"🔗 https://app.evidently.cloud/projects/{project.id}/dashboard")

✅ Dashboard saved to Evidently Cloud
🔗 https://app.evidently.cloud/projects/019d7204-9bfc-76e9-8c54-ee541e230e6f/dashboard


In [19]:
from evidently.ui.workspace import Workspace

# ── 1. Connect to local workspace ──────────────────────────────────────────
local_ws = Workspace.create("./evidently_workspace")

# ── 2. Check what projects actually exist locally ──────────────────────────
local_projects = local_ws.list_projects()
print(f"Local projects: {len(local_projects)}")
for p in local_projects:
    print(f"  Name: {p.name}, ID: {p.id}")

Local projects: 4
  Name: LLM Audit, ID: 019d75dd-1aa2-702d-9fe2-bc40f68caf49
  Name: LLM Audit, ID: 019d75e0-4a18-749e-a3a0-169182d86cec
  Name: LLM Audit, ID: 019d75e2-4b0f-7b9d-84fc-98e156cc524e
  Name: LLM Audit, ID: 019d75e0-da8c-7f52-9f1f-cf5ac080fa90


In [20]:
# ── 3. Create fresh local project ─────────────────────────────────────────
local_project = local_ws.create_project("LLM Audit")
local_project.description = "Project for auditing LLM performance on models, data, and LLMs evaluations."
local_project.save()
print(f"✅ Local project created: {local_project.id}")

# ── 4. Run report ──────────────────────────────────────────────────────────
report = Report(
    metrics=[TextEvals(columns=["answer"])],
    tags=["LLM-Audit", "groq", "Banking"]
)
snapshot = report.run(current_data=eval_dataset)
print("✅ Report ran")

# ── 5. Upload snapshot locally ─────────────────────────────────────────────
local_ws.add_run(local_project.id, snapshot)
print("✅ Snapshot uploaded locally")

# ── 6. Print correct local URL ─────────────────────────────────────────────
print(f"\n🔗 Go to: http://localhost:8080")
print(f"   Then click 'LLM Audit' project")
print(f"   Local project ID: {local_project.id}")

✅ Local project created: 019d75e4-50a0-7e76-826d-f37abd835f9f
✅ Report ran
✅ Snapshot uploaded locally

🔗 Go to: http://localhost:8080
   Then click 'LLM Audit' project
   Local project ID: 019d75e4-50a0-7e76-826d-f37abd835f9f
